# Lab 60 (solution): Multimodal RAG, runnable

Reference implementation. The two architectures from [concepts/rag/multimodal-rag.md](../../../concepts/rag/multimodal-rag.md) - shared-space (CLIP/SigLIP) vs caption-then-embed - and the 'CLIP can't read' failure, made runnable with deterministic stand-in embedders.

## Step 0: Setup

In [ ]:
from multimodal import (SharedSpaceEmbedder, CaptionThenEmbedder, recall_at_1,
                       grounded_answer, VISUAL_QUERIES, TEXT_IN_IMAGE_QUERIES, CORPUS)
shared, caption = SharedSpaceEmbedder(), CaptionThenEmbedder()
# A small corpus where each document has visual content AND dense in-image text (a number in a
# chart). Two architectures index it; the queries split into "what it looks like" vs "what it says".
print(f"{len(CORPUS)} multimodal documents; SharedSpace (CLIP-style) vs CaptionThenEmbed")

## Step 1: Visual queries - both architectures retrieve

In [ ]:
# Visual queries ("blue bar chart") - both architectures retrieve well, because both encode the
# visual content.
print(f"visual queries recall@1:  shared {recall_at_1(shared, VISUAL_QUERIES):.2f}   caption {recall_at_1(caption, VISUAL_QUERIES):.2f}")

## Step 2: Text-in-image queries - the 'CLIP can't read' failure

In [ ]:
# Text-in-image queries ("quarterly revenue") - the answer is a number printed inside the figure.
# The shared-space embedder never encoded it (the famous "CLIP can't read" failure); caption-then-
# embed ran OCR into the index, so it retrieves correctly.
print(f"text-in-image recall@1:   shared {recall_at_1(shared, TEXT_IN_IMAGE_QUERIES):.2f} (can't read)   caption {recall_at_1(caption, TEXT_IN_IMAGE_QUERIES):.2f}")

## Step 3: Retrieval vs grounding are different metrics

In [ ]:
# Retrieval and grounding are different metrics. Even if shared space retrieved the right figure,
# it cannot return the number it never encoded; the caption path can. Multimodal eval must separate
# "did the right element come back" from "did the answer read the image correctly".
print("query: 'what was Q4 revenue'")
print(f"  shared-space answer:   {grounded_answer(shared, 'what was Q4 revenue')}   (never saw the number)")
print(f"  caption-then-embed:    {grounded_answer(caption, 'what was Q4 revenue')}")

## What you built

A runnable version of the two multimodal-RAG architectures and the failure that separates them. `SharedSpaceEmbedder` encodes only what an image looks like (a CLIP/SigLIP stand-in); `CaptionThenEmbedder` runs a captioner that also reads the dense in-image text into the index. On visual queries both retrieve well (recall@1 1.00); on text-in-image queries the shared-space embedder scores 0.00 - it never encoded the numbers - while caption-then-embed recovers them (1.00). And `grounded_answer` shows that retrieval and grounding are different metrics: the shared-space path can't return a number it never saw even when it retrieves the right figure.

**Where this simplifies:** the embedders are deterministic token-overlap stand-ins and the corpus is tiny, so the lab runs offline and the 'CLIP can't read' effect is engineered by keeping the visual and in-image-text vocabularies disjoint. A real system uses a vision-language embedder and generator, where the same failure is real but graded rather than guaranteed - which is why you build an eval set with answers that live *only* in the image, and report retrieval, grounding, and OCR-reading separately. Concept: [concepts/rag/multimodal-rag.md](../../../concepts/rag/multimodal-rag.md).